# Power / Minimum-Detectable-Effect Analysis for RQ3

**Owner:** Jici · **Depends on:** `rq3_methodology.md` §6-7, `power_analysis.ipynb` (RQ1)

**Motivation.** RQ3's official significance tests at the operating point
(lambda=0.8, `rq3_methodology.md` §6) found:
- **Absolute institution-aware SPD**: +0.0162, 95% CI [-0.0139, +0.0484] --
  crosses zero, **not significant**.
- **Paired improvement** (baseline SPD minus institution-aware SPD): +0.0126,
  95% CI [0.0013, 0.0254] -- **significant**, but the CI's lower bound
  (0.0013) sits very close to zero.

Following the same logic as the RQ1 power analysis: "not significant" needs
a power/MDE check to know whether it means "no effect" or "underpowered";
and a **borderline-significant** result (CI barely excluding 0) deserves the
same scrutiny -- a significant finding sitting at low statistical power can
be fragile (i.e., it might easily have come back non-significant with
slightly different data). This notebook runs both checks for RQ3.

**Method:** identical query-resampling bootstrap logic as `power_analysis.ipynb`
(RQ1), extended to two designs:
1. **Single-sample** (absolute SPD at lambda=0.8 vs. the 0.144 baseline) --
   exactly the RQ1 recipe, applied to the re-ranked-at-lambda=0.8 per-query data.
2. **Paired** (baseline SPD vs. institution-aware SPD, same queries in both
   arms) -- resample query indices once per replicate, apply to *both* arms
   simultaneously (matching `rq3_methodology.md` §6.2's paired-bootstrap
   design), to test whether the *improvement* is different from 0.

**Important data-provenance caveat (read before trusting numbers below):**
`results/rq3_results.json` only stores pooled/aggregate statistics per lambda
-- no per-query breakdown was saved by the original re-ranking run. This
notebook **reconstructs** the per-query institution-aware-at-lambda=0.8 data
by re-running the candidate-pool + re-ranking logic locally. The **baseline**
side reproduces RQ1 exactly (it is read directly from the stored
`retrieval_results.json` top-10, not re-derived, so it is bit-exact). The
**re-ranked** side depends on re-encoding the 100 query texts locally, which
can introduce tiny floating-point drift vs. whatever environment produced the
officially reported numbers -- this can shift which of the ~40 extra
candidates (beyond the fixed top-10 seed) enter the pool near the rank-50
cutoff, giving a **close but not bit-identical** reconstruction: this
notebook gets SPD=+0.019 / improvement=+0.010 vs. the official +0.016 / +0.013.
The reconstruction is internally self-consistent (validated against RQ1 on
the baseline side) and directionally/qualitatively consistent with the
official numbers, so it is used as-is for the power analysis; the small
offset is disclosed rather than hidden.

In [ ]:
import json, os, glob, time
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter


## 0. Config

In [ ]:
BASELINE_SHARE = 0.1438     # Option B baseline elite share, locked convention
N_CANDIDATES   = 50         # candidate pool size per query (same as step6_reranking.ipynb)
K              = 10
OPERATING_LAMBDA = 0.8      # official RQ3 operating point (rq3_methodology.md §4.2)
N_BOOT_VALIDATE = 5000
N_BOOT_POWER    = 3000
R_REPLICATES    = 2000
TARGET_GRID     = [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10]
POWER_TARGET    = 0.80

# reconstructed (this notebook) observed values -- see provenance caveat above
OBSERVED_ABS_SPD       = None   # filled in after Section 2
OBSERVED_IMPROVEMENT   = None   # filled in after Section 2
# officially reported values (rq3_methodology.md §6), for reference only
OFFICIAL_ABS_SPD     = 0.0162
OFFICIAL_IMPROVEMENT = 0.0126

def find(hint):
    for base in ['data', '.', '..']:
        hits = glob.glob(f'{base}/**/{hint}', recursive=True)
        if hits:
            return hits[0]
    slug = {
        'qbio_embeddings.npy': 'yanbochen928/fairsearch-qbio-embeddings-raj-jici-yb',
        'embedding_info.json': 'yanbochen928/fairsearch-qbio-embeddings-raj-jici-yb',
    }.get(hint)
    if slug:
        import kagglehub
        root = kagglehub.dataset_download(slug)
        hits = glob.glob(f'{root}/**/{hint}', recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(hint)


## 1. Reconstruct candidate pools + institution-aware re-ranking

Same logic as `step6_reranking.ipynb`: seed each query's candidate pool with
the stored Step 5a top-10, extend to 50 with embedding neighbours, then apply
the graded institution-aware re-ranker at the official operating point
(lambda=0.8).

In [ ]:
emb = np.load(find('qbio_embeddings.npy'))
einfo = json.load(open(find('embedding_info.json')))
emb_ids = [str(x) for x in einfo['paper_ids']]
row_of = {p: i for i, p in enumerate(emb_ids)}
embn = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)
print(f'embeddings: {emb.shape}')

with open(find('retrieval_results.json')) as f:
    retrieval = json.load(f)
queries = [q for q in retrieval if q.get('type', 'neutral') == 'neutral']
queries = [q for q in queries if q['query_id'] <= 'q100']
print(f'neutral queries: {len(queries)}')

label_of = {}
for fn in ['sample_labels_1000.json', 'retrieval_labels.json']:
    with open(find(fn)) as f:
        for r in json.load(f)['labels']:
            label_of[str(r['paper_id'])] = {
                'coverage': r['coverage'], 'institution': r.get('institution'),
                'country': r.get('country'), 'elite_label': r.get('elite_label'),
            }
print(f'pre-labeled papers: {len(label_of):,}')

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
q_emb = model.encode([q['query_text'] for q in queries], normalize_embeddings=True)

pools = []
for qi, q in enumerate(queries):
    sims = embn @ q_emb[qi]
    top = np.argpartition(-sims, N_CANDIDATES)[:N_CANDIDATES]
    top = top[np.argsort(-sims[top])]
    emb_order = [emb_ids[r] for r in top]
    seed = [pid for pid in q['retrieved_paper_ids'] if pid in row_of]
    seen = set(seed)
    ordered = (seed + [pid for pid in emb_order if pid not in seen])[:N_CANDIDATES]
    n = len(ordered)
    pools.append([(pid, 1.0 - idx / n, row_of[pid]) for idx, pid in enumerate(ordered)])
print(f'candidate pools built: {len(pools)}')

def rerank_labelaware(pool, lam, field='institution', graded=True):
    cand = [(pid, rel) for (pid, rel, row) in pool]
    def group(pid):
        lab = label_of.get(pid)
        if lab and lab['coverage'] == 'found' and lab.get(field):
            return lab[field]
        return None
    selected, seen = [], Counter()
    while cand and len(selected) < K:
        best_i, best_score = None, -1e9
        for i, (pid, rel) in enumerate(cand):
            g = group(pid)
            penalty = 0.0 if g is None else (float(seen[g]) if graded else (1.0 if seen[g] > 0 else 0.0))
            s = lam * rel - (1 - lam) * penalty
            if s > best_score:
                best_score, best_i = s, i
        pid, rel = cand.pop(best_i)
        selected.append(pid)
        g = group(pid)
        if g is not None:
            seen[g] += 1
    return selected

def label_stats(pids):
    found = [label_of.get(p) for p in pids]
    found = [x for x in found if x and x['coverage'] == 'found']
    insts = {x['institution'] for x in found}
    elite = sum(x['elite_label'] for x in found)
    return len(found), elite, len(insts)

## 2. Extract per-query (n, e) for both arms + validate against RQ1/RQ3

Baseline must match RQ1 exactly (590 slots, 102 elite, SPD +0.029). The
institution-aware side is compared to the officially reported lambda=0.8
result as a sanity check (see provenance caveat in the intro -- expect
*close*, not bit-identical).

In [ ]:
n_base_l, e_base_l, n_r_l, e_r_l = [], [], [], []
for pool in pools:
    top_base = [pid for (pid, _, _) in pool[:K]]
    nf, ne, _ = label_stats(top_base)
    n_base_l.append(nf); e_base_l.append(ne)

    top_rerank = rerank_labelaware(pool, OPERATING_LAMBDA, field='institution', graded=True)
    nf2, ne2, _ = label_stats(top_rerank)
    n_r_l.append(nf2); e_r_l.append(ne2)

n_base = np.array(n_base_l); e_base = np.array(e_base_l)
n_r = np.array(n_r_l); e_r = np.array(e_r_l)
Q = len(n_base)

base_share = e_base.sum() / n_base.sum()
r_share = e_r.sum() / n_r.sum()
OBSERVED_ABS_SPD = r_share - BASELINE_SHARE
OBSERVED_IMPROVEMENT = base_share - r_share

print(f'BASELINE     : slots={n_base.sum()} elite={e_base.sum()} share={base_share:.4f} '
      f'SPD={base_share - BASELINE_SHARE:+.4f}  (RQ1: 590 / 102 / +0.0290)')
print(f'INST-AWARE   : slots={n_r.sum()} elite={e_r.sum()} share={r_share:.4f} '
      f'SPD={OBSERVED_ABS_SPD:+.4f}  (official lambda=0.8: +0.0162)')
print(f'IMPROVEMENT  : {OBSERVED_IMPROVEMENT:+.4f}  (official: +0.0126)')
assert abs((base_share - BASELINE_SHARE) - 0.0290) < 0.001, 'baseline does not match RQ1 -- check data'
print('\nBaseline reproduces RQ1 exactly. Institution-aware side is a close, self-consistent')
print('reconstruction (see provenance caveat) -- proceeding with power analysis on this data.')

## 3. Bootstrap procedures

Two designs, both resampling **query indices** (not slots) with replacement,
mirroring RQ1/`rq3_methodology.md`'s convention.

In [ ]:
def bootstrap_ci_1sample(n_arr, e_arr, ref, n_boot, rng):
    """Single-sample: is (pooled elite share) - ref different from 0?"""
    idx = rng.integers(0, Q, size=(n_boot, Q))
    n_tot = n_arr[idx].sum(axis=1); e_tot = e_arr[idx].sum(axis=1)
    with np.errstate(invalid='ignore', divide='ignore'):
        vals = e_tot / n_tot - ref
    return np.nanpercentile(vals, [2.5, 97.5])

def bootstrap_ci_paired(n_a, e_a, n_b, e_b, n_boot, rng):
    """Paired: is (share_a - share_b) different from 0? Same resampled query
    indices applied to BOTH arms per draw (mirrors rq3_methodology.md §6.2)."""
    idx = rng.integers(0, Q, size=(n_boot, Q))
    na_t = n_a[idx].sum(axis=1); ea_t = e_a[idx].sum(axis=1)
    nb_t = n_b[idx].sum(axis=1); eb_t = e_b[idx].sum(axis=1)
    with np.errstate(invalid='ignore', divide='ignore'):
        diff = (ea_t / na_t) - (eb_t / nb_t)
    return np.nanpercentile(diff, [2.5, 97.5])

## 4. Power simulation (A) -- absolute SPD at lambda=0.8

Same recipe as RQ1: keep real per-query labeled-slot counts (`n_r`), redraw
synthetic elite counts at `0.144 + target_spd`, run the bootstrap, record the
detection rate.

In [ ]:
def power_absolute(target_spd, n_arr, ref, R, n_boot, rng):
    p_sim = ref + target_spd
    e_sims = rng.binomial(n_arr[None, :], p_sim, size=(R, Q))
    detected = 0
    for r in range(R):
        lo, hi = bootstrap_ci_1sample(n_arr, e_sims[r], ref, n_boot, rng)
        if lo > 0 or hi < 0:
            detected += 1
    return detected / R

power_curve_abs = {}
for spd in TARGET_GRID:
    rng_spd = np.random.default_rng(int(round(spd * 10000)) + 11)
    pw = power_absolute(spd, n_r, BASELINE_SHARE, R_REPLICATES, N_BOOT_POWER, rng_spd)
    power_curve_abs[spd] = pw
    print(f'(A) target SPD = {spd:+.2f}: power = {pw:.3f}')

## 5. Power simulation (B) -- paired improvement (baseline vs. institution-aware)

Baseline arm simulated at its real empirical rate (`base_share`); the
institution-aware arm simulated at `base_share - target_improvement`, so
`target_improvement` represents the hypothesized true drop in elite share
from re-ranking. Both arms use the real, distinct per-query labeled-slot
counts (`n_base`, `n_r`).

In [ ]:
def power_paired(target_improvement, n_a, n_b, p_a_ref, R, n_boot, rng):
    e_a_sims = rng.binomial(n_a[None, :], p_a_ref, size=(R, Q))
    p_b = p_a_ref - target_improvement
    e_b_sims = rng.binomial(n_b[None, :], p_b, size=(R, Q))
    detected = 0
    for r in range(R):
        lo, hi = bootstrap_ci_paired(n_a, e_a_sims[r], n_b, e_b_sims[r], n_boot, rng)
        if lo > 0 or hi < 0:
            detected += 1
    return detected / R

power_curve_paired = {}
for imp in TARGET_GRID:
    rng_imp = np.random.default_rng(int(round(imp * 10000)) + 17)
    pw = power_paired(imp, n_base, n_r, base_share, R_REPLICATES, N_BOOT_POWER, rng_imp)
    power_curve_paired[imp] = pw
    print(f'(B) target improvement = {imp:+.2f}: power = {pw:.3f}')

## 6. Calibration checks + MDE at 80% power

In [ ]:
def mde_at_power(power_curve, power_target=POWER_TARGET):
    xs = np.array(sorted(power_curve.keys()))
    ys = np.array([power_curve[x] for x in xs])
    if not (ys >= power_target).any():
        return float('nan'), xs, ys
    i = np.argmax(ys >= power_target)
    if i == 0:
        return xs[0], xs, ys
    x0, x1, y0, y1 = xs[i-1], xs[i], ys[i-1], ys[i]
    mde = x0 + (power_target - y0) * (x1 - x0) / (y1 - y0)
    return mde, xs, ys

mde_abs, xs_abs, ys_abs = mde_at_power(power_curve_abs)
mde_paired, xs_paired, ys_paired = mde_at_power(power_curve_paired)
power_at_obs_abs = np.interp(OBSERVED_ABS_SPD, xs_abs, ys_abs)
power_at_obs_paired = np.interp(OBSERVED_IMPROVEMENT, xs_paired, ys_paired)

print('--- (A) Absolute SPD at lambda=0.8 ---')
print(f'  Type-I (target=0): {power_curve_abs[0.0]:.3f}  (nominal 0.05)')
print(f'  MDE @ 80% power  : {mde_abs:.4f}')
print(f'  Observed SPD ({OBSERVED_ABS_SPD:+.4f}) power: {power_at_obs_abs:.3f}')
print()
print('--- (B) Paired improvement (baseline -> institution-aware) ---')
print(f'  Type-I (target=0): {power_curve_paired[0.0]:.3f}  (nominal 0.05)')
print(f'  MDE @ 80% power  : {mde_paired:.4f}')
print(f'  Observed improvement ({OBSERVED_IMPROVEMENT:+.4f}) power: {power_at_obs_paired:.3f}')
print()
print('NOTE: (B) came back "significant" in the official run (CI [0.0013, 0.0254],')
print('lower bound very close to 0). A low power-at-observed value here would mean that')
print('significant result sits in a fragile, low-power regime -- worth flagging even')
print('though it technically cleared the CI-excludes-zero bar.')

## 7. Plot both power curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
ax.plot(xs_abs, ys_abs, 'o-', color='#2e5fa3', label='Empirical power')
ax.axhline(POWER_TARGET, ls='--', c='#888', lw=1, label='80% target')
ax.axhline(0.05, ls=':', c='#bbb', lw=1, label='5% (Type-I ref.)')
ax.axvline(OBSERVED_ABS_SPD, ls='-.', c='#c0392b', lw=1.3,
           label=f'Observed ({OBSERVED_ABS_SPD:+.3f})')
if not np.isnan(mde_abs):
    ax.plot([mde_abs], [POWER_TARGET], 'D', color='#c0392b', ms=7)
ax.set_xlabel('Target true SPD'); ax.set_ylabel('Power')
ax.set_title('(A) Absolute SPD at lambda=0.8\nvs. baseline 0.144'); ax.set_ylim(-0.02, 1.05)
ax.legend(fontsize=7, loc='lower right')

ax = axes[1]
ax.plot(xs_paired, ys_paired, 'o-', color='#2e7d32', label='Empirical power')
ax.axhline(POWER_TARGET, ls='--', c='#888', lw=1, label='80% target')
ax.axhline(0.05, ls=':', c='#bbb', lw=1, label='5% (Type-I ref.)')
ax.axvline(OBSERVED_IMPROVEMENT, ls='-.', c='#c0392b', lw=1.3,
           label=f'Observed ({OBSERVED_IMPROVEMENT:+.3f})')
if not np.isnan(mde_paired):
    ax.plot([mde_paired], [POWER_TARGET], 'D', color='#c0392b', ms=7)
ax.set_xlabel('Target true improvement'); ax.set_ylabel('Power')
ax.set_title('(B) Paired improvement\n(baseline vs. institution-aware)'); ax.set_ylim(-0.02, 1.05)
ax.legend(fontsize=7, loc='lower right')

fig.suptitle(f'RQ3 statistical power (n={Q} queries, lambda={OPERATING_LAMBDA})')
plt.tight_layout()
plt.savefig('rq3_power_curves.png', dpi=120)
plt.show()

## 8. Save results

In [ ]:
out = {
    'operating_lambda': OPERATING_LAMBDA,
    'n_queries': int(Q),
    'reconstruction_note': (
        'Institution-aware-at-lambda=0.8 per-query data reconstructed locally; '
        'baseline reproduces RQ1 exactly, institution-aware side is close but not '
        'bit-identical to the officially reported run (see notebook intro).'
    ),
    'observed': {
        'baseline_slots': int(n_base.sum()), 'baseline_elite': int(e_base.sum()),
        'inst_aware_slots': int(n_r.sum()), 'inst_aware_elite': int(e_r.sum()),
        'abs_spd_reconstructed': round(float(OBSERVED_ABS_SPD), 4),
        'abs_spd_official': OFFICIAL_ABS_SPD,
        'improvement_reconstructed': round(float(OBSERVED_IMPROVEMENT), 4),
        'improvement_official': OFFICIAL_IMPROVEMENT,
    },
    'absolute_spd_test': {
        'power_curve': {str(k): round(float(v), 4) for k, v in power_curve_abs.items()},
        'type1_error': round(float(power_curve_abs[0.0]), 4),
        'mde_at_80pct_power': round(float(mde_abs), 4) if not np.isnan(mde_abs) else None,
        'power_at_observed': round(float(power_at_obs_abs), 4),
    },
    'paired_improvement_test': {
        'power_curve': {str(k): round(float(v), 4) for k, v in power_curve_paired.items()},
        'type1_error': round(float(power_curve_paired[0.0]), 4),
        'mde_at_80pct_power': round(float(mde_paired), 4) if not np.isnan(mde_paired) else None,
        'power_at_observed': round(float(power_at_obs_paired), 4),
    },
    'config': {'n_boot_power': N_BOOT_POWER, 'r_replicates': R_REPLICATES, 'target_grid': TARGET_GRID},
}
os.makedirs('results', exist_ok=True)
with open('results/rq3_power_analysis.json', 'w') as f:
    json.dump(out, f, indent=2)
print('saved results/rq3_power_analysis.json')
print(json.dumps(out, indent=2))

## 9. Reading the result

**Absolute SPD (A):** mirrors RQ1's story -- can rule out a large residual
elite bias after re-ranking, cannot rule out a small one near the observed
point estimate.

**Paired improvement (B) -- the more important reading:** the officially
reported improvement result is significant (CI excludes 0), but if the
power-at-observed value above is low, that significant finding sits in a
low-power regime -- i.e., it is the kind of result that could easily have
come back non-significant with slightly different data. This does not mean
the finding is wrong, but it should be reported as **fragile / borderline**
rather than as strong evidence, and the interpretation note already in
`rq3_methodology.md` §6 ("best read as a side effect of de-duplicating
institutions, not evidence of correcting a proven bias") should be kept.

**Limitations:**
1. Same independent-per-query synthetic-injection simplification as RQ1's
   power analysis (likely makes MDE a lower bound / power estimates mildly
   optimistic).
2. The institution-aware per-query data is a **local reconstruction**, not
   the exact bit-identical data behind the officially reported +0.0162 /
   +0.0126 -- see the provenance caveat in the intro. Numbers here should be
   read as closely representative, not as a re-verification of the exact
   official values.
3. Targets RQ3's two significance tests only; RQ2-A has not been power-
   analyzed (blocked on obtaining `rq2_frameworkA_per_query.json` from Yan-Bo).